### This notebook will contain a variant of the Rocket Richard Baseline Predictor made in rr1.ipynb, that is, this variant will be using General Skater Stats (GSS) + EDGE Stats
#### This is done for the sake of model fine tuning and evaluating performance

In [ ]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import ast
from helpersrr import clear_csv, extractPlayerID, placeToStats, fetchSkaterStats, labelWinners, formatEdgeStats

import importlib
import helpersrr
importlib.reload(helpersrr)

<module 'helpersrr' from '/Users/Joaquin/Documents/GitHub/nhl-trophy-predictor/notebooks/helpersrr.py'>

In [129]:
#global variables
ranks = False
client = NHLClient()
rocket_richard = clear_csv("../data/formattedwebscraped/rocketrichard.csv")

first_place = rocket_richard[['szn','winner']]
first_ids = placeToStats(first_place)
second_place = rocket_richard[['szn', 'runner_up']]
second_ids = placeToStats(second_place)
third_place = rocket_richard[['szn', 'finalist']]
third_ids = placeToStats(third_place)

../data/formattedwebscraped/rocketrichard.csv


### Milestone 1: Get EDGE Statistics from the API and place them in data/api/EDGEstats

In [ ]:
def fetchSkaterStats(year, csv=False, edge=False):      #optimized version of fetchSkaterStats
    if len(year) == 4:
        year_interval = f"{int(year)}{int(year)+1}"
    elif len(year) == 8:
        year_interval = year
    else:
        raise SyntaxError("requires yyyy or yyyyyyyy format of season year")

    page_size = 100
    rows = []

    if edge:
        for start in range(0, 1000, page_size):
            statChunk = client.stats.skater_stats_summary(
                start_season=year_interval,
                end_season=year_interval,
                start=start,
                limit=page_size
            )
            if not statChunk:
                break

            ids = pd.DataFrame(statChunk)['playerId'].tolist()
            for player_id in ids:
                try:
                    individual_stat = client.edge.skater_detail(
                        player_id=player_id,
                        season=year_interval
                    )
                    rows.append(formatEdgeStats(individual_stat, shotDetails=True))
                except Exception as exc:
                    print("edge fetch failed", player_id, exc)

        df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    else:
        for start in range(0, 1000, page_size):
            statChunk = client.stats.skater_stats_summary(
                start_season=year_interval,
                end_season=year_interval,
                start=start,
                limit=page_size
            )
            if not statChunk:
                break
            rows.extend(statChunk)

        df = pd.DataFrame(rows)

    if csv:
        out = f"../data/api/EDGEstats/skatersEDGE{year_interval}.csv" if edge else f"../data/api/skaters/skaters{year_interval}.csv"
        df.to_csv(out, index=False)
    else:
        return df

In [ ]:
#DO NOT RUN THIS IF COMMENTED -- essentially places relevant EDGE stats files into the data/api/EDGEstats folder
#edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
#fetchSkaterStats('20212022',csv=True,edge=True)
#for season in edgeSzns:
#    fetchSkaterStats(season, csv=True, edge=True)
#    print(f"done season {season}")

edge fetch failed 8477073 'topShotSpeed'
edge fetch failed 8478906 'topShotSpeed'
edge fetch failed 8476495 'topShotSpeed'
edge fetch failed 8476870 'topShotSpeed'
edge fetch failed 8477631 'topShotSpeed'
edge fetch failed 8480913 'topShotSpeed'
edge fetch failed 8482408 'topShotSpeed'


In [ ]:
#this cell is where formatEdgeStats() was originally made
cm97 = client.edge.skater_detail(player_id='8478402', season='20252026')
formttedcm97 = formatEdgeStats(cm97, True)
formttedcm97


,playerId,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,...,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots
0,8478402,132.0467,39.6089,531.4875,7.908,8,1,92,9,120,...,34,3,97,3,15,9,29,0,12,1


In [ ]:
edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
#will have to fetch stats for each player in each season, but after preprocessing
df = pd.DataFrame(client.stats.skater_stats_summary(
    start_season="20252026",
    end_season="20252026"))

ids = df['playerId']
new_df = []
for id in ids:
    ind_stat = client.edge.skater_detail(player_id=id)
    formatted_stat = formatEdgeStats(ind_stat, shotDetails=True)
    new_df.append(formatted_stat)



[('Behind the Net', 11), ('Beyond Red Line', 1), ('Center Point', 4), ('Crease', 23), ('High Slot', 27), ('L Circle', 36), ('L Corner', 1), ('L Net Side', 34), ('L Point', 3), ('Low Slot', 97), ('Offensive Neutral Zone', 3), ('Outside L', 15), ('Outside R', 9), ('R Circle', 29), ('R Corner', 0), ('R Net Side', 12), ('R Point', 1)]
[('Behind the Net', 2), ('Beyond Red Line', 1), ('Center Point', 7), ('Crease', 5), ('High Slot', 25), ('L Circle', 16), ('L Corner', 2), ('L Net Side', 6), ('L Point', 9), ('Low Slot', 40), ('Offensive Neutral Zone', 2), ('Outside L', 6), ('Outside R', 40), ('R Circle', 37), ('R Corner', 0), ('R Net Side', 11), ('R Point', 22)]
[('Behind the Net', 3), ('Beyond Red Line', 3), ('Center Point', 16), ('Crease', 11), ('High Slot', 42), ('L Circle', 77), ('L Corner', 1), ('L Net Side', 19), ('L Point', 11), ('Low Slot', 72), ('Offensive Neutral Zone', 5), ('Outside L', 38), ('Outside R', 12), ('R Circle', 30), ('R Corner', 0), ('R Net Side', 5), ('R Point', 5)]
[(

In [ ]:
#experimenting how to merge the GSS + EDGE stats (this is meant for milestone 2)

final_df = pd.DataFrame()
for item in new_df:
    final_df = pd.concat([final_df,item])
regularDf = fetchSkaterStats("20252026", csv=False, edge=False)
regularDf = regularDf.merge(final_df)

#with pd.option_context("display.max_rows", None, "display.max_columns", None):
#        display(regularDf)

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,lastName,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,positionCode,ppGoals,ppPoints,seasonId,shGoals,shPoints,shootingPct,shootsCatches,shots,skaterFullName,teamAbbrevs,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longGoalPctg,midGoalPctg,highGoalPctg,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,Behind the Net Shots,Beyond Red Line Shots,Center Point Shots,Crease Shots,High Slot Shots,L Circle Shots,L Corner Shots,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots
0,90,34,82,0.49512,4,82,48,McDavid,1,44,8478402,17,138,1.68292,C,13,54,20252026,1,2,0.15686,L,306,Connor McDavid,EDM,1379.1219,132.0467,39.6089,531.4875,7.9080,0.125000,0.097826,0.216667,0.476879,0.169191,0.353930,11,1,4,23,27,36,1,34,3,97,3,15,9,29,0,12,1
1,86,35,92,0.00000,8,76,44,Kucherov,2,50,8476453,43,130,1.71052,R,8,37,20252026,1,1,0.19047,L,231,Nikita Kucherov,TBL,1220.1052,148.1562,36.1133,392.7248,7.2929,0.078947,0.153846,0.400000,0.456801,0.185946,0.357253,2,1,7,5,25,16,2,6,9,40,2,6,40,37,0,11,22
2,74,42,97,0.51049,7,80,53,MacKinnon,1,39,8477492,57,127,1.58750,C,11,30,20252026,0,0,0.15142,R,350,Nathan MacKinnon,COL,1335.7125,146.4342,39.8786,456.0059,7.8713,0.187500,0.134228,0.228916,0.466145,0.180970,0.352885,3,3,16,11,42,77,1,19,11,72,5,38,12,30,0,5,5
3,70,37,82,0.48638,5,82,45,Celebrini,2,44,8484801,8,115,1.40243,C,8,33,20252026,0,0,0.15679,L,287,Macklin Celebrini,SJS,1278.9512,157.5387,35.9208,472.6669,7.5709,0.107143,0.157233,0.220339,0.445342,0.181240,0.373417,2,2,20,5,66,52,0,4,3,54,5,17,6,41,0,5,5
4,67,29,80,0.46661,6,82,36,Scheifele,1,43,8476460,0,103,1.25609,C,7,22,20252026,0,1,0.20571,R,175,Mark Scheifele,WPG,1290.2439,139.8681,37.7198,467.6965,7.1238,0.250000,0.257143,0.166667,0.439736,0.176693,0.383570,3,1,3,6,20,31,1,11,0,60,2,7,8,19,1,1,1
5,72,17,57,0.50448,2,82,29,Suzuki,1,28,8480018,37,101,1.23170,C,11,43,20252026,1,1,0.15846,R,183,Nick Suzuki,MTL,1249.2317,138.4358,38.0149,437.0927,6.7465,0.000000,0.136364,0.192308,0.451094,0.173137,0.375769,2,3,5,6,20,29,1,8,3,46,3,18,14,17,2,3,3
6,71,19,67,0.22222,4,77,29,Pastrnak,1,72,8477956,4,100,1.29870,R,10,33,20252026,0,0,0.11111,R,261,David Pastrnak,BOS,1239.3376,149.7656,34.6180,398.6906,6.8550,0.121951,0.088235,0.175439,0.435708,0.184081,0.380211,3,3,14,8,29,29,0,13,15,49,7,19,11,44,2,3,12
7,62,29,76,0.42857,5,78,38,Necas,0,30,8480039,47,100,1.28205,C,9,24,20252026,0,0,0.18446,R,206,Martin Necas,COL,1289.8846,150.2805,38.4498,466.3214,8.1924,0.080000,0.166667,0.222222,0.454715,0.181042,0.364242,0,3,11,5,14,37,0,8,7,49,2,10,16,33,0,4,7
8,62,18,54,0.56935,3,65,35,Draisaitl,1,26,8477934,13,97,1.49230,C,16,42,20252026,1,1,0.18817,L,186,Leon Draisaitl,EDM,1294.5076,138.8220,37.4597,343.2309,6.7380,0.000000,0.244898,0.283019,0.466764,0.173188,0.360048,1,2,5,5,21,5,1,5,0,48,3,7,26,23,0,29,5
9,51,30,55,0.33333,9,82,45,Robertson,1,32,8480027,22,96,1.17073,L,15,41,20252026,0,0,0.15306,L,294,Jason Robertson,DAL,1215.1829,145.1306,36.4681,413.6000,6.4472,0.129032,0.112245,0.212963,0.455439,0.175916,0.368645,2,3,14,13,32,41,1,16,8,95,7,12,9,25,0,7,9


### Milestone 2: Make the new feature set (GSS + EDGE Stats)

### Milestone 3: Train model

### Milestone 4: Test on Skater Stats from 2023-2026 Season

In [ ]:
#do 3 seperate tests: 1 for 2023-2024, 1 for 2024-2025, 1 for 2025-2026
#2023-2024
'''
test2023 = testingSets[2]
if ranks == False:
    test_x = test2023.drop(columns=['lastName','rrWinner','playerId','seasonId'])
    test_y = test2023['rrWinner']
    print("only one winner")
else:
    test_x = test2023.drop(columns=['lastName','rrRank','playerId','seasonId'])
    test_y = test2023['rrRank']
    print("top3")

pred_y = model.predict(test_x)
predictions = pd.Series(pred_y)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = test2023.join(predictions)
'''


'\ntest2023 = testingSets[2]\nif ranks == False:\n    test_x = test2023.drop(columns=[\'lastName\',\'rrWinner\',\'playerId\',\'seasonId\'])\n    test_y = test2023[\'rrWinner\']\n    print("only one winner")\nelse:\n    test_x = test2023.drop(columns=[\'lastName\',\'rrRank\',\'playerId\',\'seasonId\'])\n    test_y = test2023[\'rrRank\']\n    print("top3")\n\npred_y = model.predict(test_x)\npredictions = pd.Series(pred_y)\npredictions = predictions.rename(\'predictions\')\npredictions = predictions.to_frame()\nresults = test2023.join(predictions)\n'